In [1]:
import os
os.chdir("..")

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from recsys_lakehouse.spark import spark_builder
from recsys_lakehouse.lakehouse.operator import TableOperator
from recsys_lakehouse.lakehouse import layers

from pathlib import Path
from pyspark.sql import functions as F, SparkSession, DataFrame
from pyspark.sql.functions import from_unixtime, col, sum, rand, when
from pyspark.sql.types import FloatType, IntegerType, TimestampType, StructType, StructField, StringType

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS

In [4]:
spark = SparkSession.builder.appName("").getOrCreate()  # type: ignore

24/12/09 19:54:14 WARN Utils: Your hostname, MacBook-Pro-Milosz.local resolves to a loopback address: 127.0.0.1; using 192.168.0.73 instead (on interface en0)
24/12/09 19:54:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/12/09 19:54:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/12/09 19:54:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
dataset_name = "amazon_books_sample10000"

operator = TableOperator(spark)
silver = layers.Silver(dataset_name=dataset_name)

In [6]:
operator.read_table(silver, table=silver.tables['books_reviews']).show()

+-------------------+--------------------+----------+-----------+------------+-----------------+--------------------+--------------------+------+----------+
|          timestamp|             user_id|      asin|parent_asin|helpful_vote|verified_purchase|               title|                text|rating|date_month|
+-------------------+--------------------+----------+-----------+------------+-----------------+--------------------+--------------------+------+----------+
|2013-05-31 08:18:08|AFWVN52MRBWOTIK7U...|1400074312| 1400074312|           1|            false|The only thing I ...|A reformer who de...|   4.0|   2013-05|
|2013-05-30 07:58:32|AFWVN52MRBWOTIK7U...|0399158243| 0399158243|           2|            false|A Love Story set ...|In the vein of Ba...|   3.0|   2013-05|
|2013-05-29 18:15:32|AFWVN52MRBWOTIK7U...|0062105620| 0062105620|           0|            false|Suspenseful & con...|Beautiful prose b...|   4.0|   2013-05|
|2013-05-29 08:42:05|AFWVN52MRBWOTIK7U...|0761453830| 0761

In [7]:
operator.read_table(silver, table=silver.tables['books_metadata']).show()

+-----------+--------------------+--------------------+--------------------+--------------------+-----+--------------+-------------+--------------------+-------------+
|parent_asin|               title|            subtitle|         description|          categories|price|average_rating|rating_number|              images|main_category|
+-----------+--------------------+--------------------+--------------------+--------------------+-----+--------------+-------------+--------------------+-------------+
| B08M2G2JTH|Needle Felting fo...|Paperback – Octob...|                  []|[Books, Crafts, H...|14.99|           3.9|           59|[{null, https://m...|        Books|
| 1481459759|Bear Can't Wait (...|Hardcover – Pictu...|[About the Author...|[Books, Children'...|10.99|           4.9|          505|[{null, https://m...|        Books|
| 1641373415|You Can't Fix Wha...|Paperback – Decem...|[Review, A fresh,...|[Books, History, ...|17.79|           4.4|           23|[{null, https://m...|       

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from recsys_lakehouse.spark import spark_builder
from recsys_lakehouse.jobs.lakehouse.silver import load_table
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.functions import from_unixtime, col, sum, rand, when
from pyspark.sql.types import FloatType, IntegerType, TimestampType

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("").getOrCreate()  # type: ignore

In [ ]:
books = load_table(spark,table_path=Path(".datalake/bronze/amazon_books_sample10000"), table_name="books")

In [ ]:
books.show()

In [ ]:
books.printSchema()

In [ ]:
books.filter(F.size(col("images")) > 0).select("images").first()

In [ ]:
books_filter.select("dates").withColumn("dates", col("dates").cast(TimestampType()))

In [ ]:
books_process = books.withColumn("dates", from_unixtime(col("timestamp") / 1000, "yyyy-MM-dd HH:mm:ss")) \
     .withColumn("rating", col("rating").cast(FloatType())) \

books_filter = books_process \
    .filter(col("rating") <= 5.0) \
    .filter(col("rating") >= 1.0) \
    .filter(col("rating").isNotNull()) \
    .filter(col("asin").isNotNull()) \
    .filter(col("user_id").isNotNull()) \
    .filter(col("helpful_vote") >= 0)


books_process.printSchema()
books_process.show()
print("size: ", books_process.count())

books_filter.printSchema()
books_filter.show()
print("size: ", books_filter.count())

In [ ]:
books

In [ ]:
from recsys_lakehouse.lakehouse.silver import BooksReviewsTable

In [ ]:
b = BooksReviewsTable(books)
df = b.process(spark)

In [ ]:
b.schema

In [ ]:
df.printSchema()

In [ ]:
books.describe("helpful_vote").show()

In [ ]:
books.groupBy("rating").count().show()

In [ ]:
books_nl = books.select([
    when(rand() < 0.1, None).otherwise(col(c)).alias(c) for c in books.columns
])

In [ ]:
null_counts = books_nl.select([sum(col(c).isNull().cast("int")).alias(c) for c in books_nl.columns])
null_counts.show()

In [ ]:
meta_books = load_table(spark,table_path=Path(".datalake/bronze/amazon_books_sample10000"), table_name="meta_books")

In [ ]:
meta_books.show()

In [ ]:
null_counts = meta_books.select([sum(col(c).isNull().cast("int")).alias(c) for c in meta_books.columns])
null_counts.show()

In [ ]:
meta_books.select("average_rating").describe("average_rating").show()

In [ ]:
meta_books.filter(col("parent_asin").isNotNull()).filter(col(""))

In [ ]:
meta_books.printSchema()